# Depth-aware Masked Inference (per-capture folder)

Each capture gets its own `analysis/depth_view/<timestamp>/` folder. The timestamp
is locked at Step 1, so every later cell writes into that same folder regardless
of when it actually runs.

| Step | Output file | What it is |
| --- | --- | --- |
| 1 | `raw.png` | plain RGB capture |
| 2 | `yolo.png` | full-frame YOLO on the raw |
| 3 | `dbscan.png` | DBSCAN cluster bboxes drawn on the RGB |
| 4 | `cluster_inference_{i}.png` | per-cluster: crop-to-bbox + non-cluster pixels blacked out + YOLO |
| 5 | `depth_aware_inference.png` | per-cluster results stitched back into the full frame |
| 6 | `3d.html` | optional plotly 3D DBSCAN scatter |


## 1. Imports

In [ ]:
import sys, time
from pathlib import Path
from datetime import datetime

import numpy as np
import cv2
import pyzed.sl as sl
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import MinMaxScaler
from ultralytics import YOLO

PROJECT_ROOT = Path('/Users/jnoael/Mechatronics_Vision_2026')
sys.path.insert(0, str(PROJECT_ROOT))

# Per-kernel state bag shared between step cells.
STATE: dict = {}
print('imports OK')


## 2. Config — the only cell you normally edit

In [ ]:
# ── Model ─────────────────────────────────────────────────────────
WEIGHTS    = PROJECT_ROOT / 'runs_cnn_transfer' / 'v1.1' / 'weights' / 'best.pt'
CONF_THRES = 0.50
IOU_THRES  = 0.50
IMGSZ      = 640
MAX_DET    = 4
CLASSES    = None
AUGMENT    = False
DEVICE     = None   # None=auto; 'cpu' forces CPU (frees GPU for ZED SDK)

# ── ZED camera ────────────────────────────────────────────────────
RESOLUTION  = sl.RESOLUTION.HD1080  # ZED X locks to HD1080/HD1200
FPS         = 5
DEPTH_MODE  = sl.DEPTH_MODE.PERFORMANCE
DEPTH_MIN_M = 0.3

# ── Shutter ───────────────────────────────────────────────────────
SHUTTER_MODE = 'single'   # 'single' | 'burst'
BURST_COUNT  = 5
FRAME_INDEX  = 0          # which burst frame to process (0 .. BURST_COUNT-1)

# ── Depth → DBSCAN ───────────────────────────────────────────────
DEPTH_DOWNSAMPLE_STEP  = 10   # every Nth pixel in X,Y before clustering
DBSCAN_EPS             = 0.03
DBSCAN_MIN_SAMPLES     = 20
MIN_CLUSTER_POINTS     = 50
MAX_CLUSTERS_PER_FRAME = 6

# ── Wall filter ──────────────────────────────────────────────────
WALL_DEPTH_M = 5.0

# ── Output ────────────────────────────────────────────────────────
ANALYSIS_DIR = PROJECT_ROOT / 'analysis' / 'depth_view'

# ── Toggle ────────────────────────────────────────────────────────
VISUALIZE_CLUSTERS_3D = False   # Step 6: plotly 3D scatter HTML

ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print(f'output root: {ANALYSIS_DIR}')
print(f'weights    : {WEIGHTS.name}')
print(f'shutter    : {SHUTTER_MODE}')


## 3. Helpers — ZED capture

In [ ]:
def open_zed() -> sl.Camera:
    zed = sl.Camera()
    init = sl.InitParameters()
    init.camera_resolution      = RESOLUTION
    init.camera_fps             = FPS
    init.depth_mode             = DEPTH_MODE
    init.coordinate_units       = sl.UNIT.METER
    init.depth_minimum_distance = DEPTH_MIN_M
    err = zed.open(init)
    if err != sl.ERROR_CODE.SUCCESS:
        raise RuntimeError(f'ZED open failed: {err}')
    res = zed.get_camera_information().camera_configuration.resolution
    print(f'  ZED opened  {res.width}x{res.height} @ {FPS} FPS  depth={DEPTH_MODE}')
    return zed


def capture_frame_and_depth(zed, runtime, image_mat, depth_mat):
    if zed.grab(runtime) != sl.ERROR_CODE.SUCCESS:
        return None, None
    zed.retrieve_image(image_mat, sl.VIEW.LEFT)
    zed.retrieve_measure(depth_mat, sl.MEASURE.DEPTH)
    rgb = image_mat.get_data()
    if rgb.ndim == 3 and rgb.shape[2] == 4:
        rgb = cv2.cvtColor(rgb, cv2.COLOR_BGRA2BGR)
    rgb   = np.ascontiguousarray(rgb)
    depth = np.ascontiguousarray(depth_mat.get_data())
    return rgb, depth


def _grab(zed, runtime, image_mat, depth_mat, retries=30):
    for _ in range(retries):
        rgb, depth = capture_frame_and_depth(zed, runtime, image_mat, depth_mat)
        if rgb is not None and depth is not None:
            return rgb, depth
    raise RuntimeError('ZED stopped providing frames')


## 4. Helpers — DBSCAN

In [ ]:
def cluster_depth(depth_m, step=None, eps=None, min_samples=None):
    step        = step        if step        is not None else DEPTH_DOWNSAMPLE_STEP
    eps         = eps         if eps         is not None else DBSCAN_EPS
    min_samples = min_samples if min_samples is not None else DBSCAN_MIN_SAMPLES
    small  = depth_m[::step, ::step]
    h, w   = small.shape
    yy, xx = np.mgrid[0:h, 0:w]
    flat   = small.ravel()
    ok     = np.isfinite(flat) & (flat > 0)
    if ok.sum() < min_samples:
        return np.empty((0, 3), dtype=np.float32), np.empty((0,), dtype=np.int64)
    points = np.column_stack([
        xx.ravel()[ok] * step,
        yy.ravel()[ok] * step,
        flat[ok],
    ]).astype(np.float32)
    points_norm = MinMaxScaler().fit_transform(points)
    labels = DBSCAN(eps=eps, min_samples=min_samples, n_jobs=-1).fit_predict(points_norm)
    return points, labels


def filter_clusters(points, labels,
                    wall_depth_m=None, min_points=None, max_clusters=None):
    wall_depth_m = wall_depth_m if wall_depth_m is not None else WALL_DEPTH_M
    min_points   = min_points   if min_points   is not None else MIN_CLUSTER_POINTS
    max_clusters = max_clusters if max_clusters is not None else MAX_CLUSTERS_PER_FRAME
    kept, dropped = [], []
    for lbl in set(labels.tolist()):
        if lbl == -1:
            continue
        mask = labels == lbl
        n    = int(mask.sum())
        xs, ys, zs = points[mask, 0], points[mask, 1], points[mask, 2]
        median_depth = float(np.median(zs))
        info = dict(label=int(lbl),
                    bbox=(int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())),
                    median_depth=median_depth, n_points=n)
        if n < min_points:
            info['reason'] = 'too_few_points'; dropped.append(info)
        elif median_depth > wall_depth_m:
            info['reason'] = 'wall'; dropped.append(info)
        else:
            kept.append(info)
    kept.sort(key=lambda c: c['n_points'], reverse=True)
    return kept[:max_clusters], dropped


def cluster_pixel_mask(points, labels, cluster_label, shape, step):
    """Rasterise a cluster's sparse points into a full-resolution binary mask.
    Each sparse point represents a `step`-px region, so we paint each one then
    dilate to fill the gaps between the downsampled samples."""
    H, W = shape[:2]
    mask = np.zeros((H, W), dtype=np.uint8)
    cpts = points[labels == cluster_label][:, :2].astype(int)
    valid = (cpts[:, 0] >= 0) & (cpts[:, 0] < W) & (cpts[:, 1] >= 0) & (cpts[:, 1] < H)
    cpts = cpts[valid]
    if len(cpts) == 0:
        return mask
    mask[cpts[:, 1], cpts[:, 0]] = 255
    kernel = np.ones((step * 2 + 1, step * 2 + 1), dtype=np.uint8)
    return cv2.dilate(mask, kernel)


## 5. Helpers — YOLO + drawing

In [ ]:
PALETTE = [(0, 255, 0), (0, 255, 255), (255, 128, 0), (255, 0, 255),
           (0, 128, 255), (128, 255, 0)]


def load_model(weights=None):
    weights = Path(weights or WEIGHTS)
    if not weights.exists():
        raise FileNotFoundError(f'weights not found: {weights}')
    print(f'  loading {weights}...')
    return YOLO(str(weights))


def predict_on_image(model, image, crop_origin=(0, 0)):
    """Run YOLO on one image. crop_origin remaps box coords to full-frame."""
    device_kw = {} if DEVICE is None else {'device': DEVICE}
    r = model.predict(image, conf=CONF_THRES, iou=IOU_THRES, imgsz=IMGSZ,
                      max_det=MAX_DET, classes=CLASSES, augment=AUGMENT,
                      verbose=False, **device_kw)[0]
    if r.boxes is None or len(r.boxes) == 0:
        return []
    xyxy   = r.boxes.xyxy.cpu().numpy()
    confs  = r.boxes.conf.cpu().numpy()
    clsids = r.boxes.cls.cpu().numpy().astype(int)
    ox, oy = crop_origin
    dets = []
    for bb, cf, k in zip(xyxy, confs, clsids):
        gx1, gy1, gx2, gy2 = bb + np.array([ox, oy, ox, oy])
        dets.append(dict(bbox=(int(gx1), int(gy1), int(gx2), int(gy2)),
                         cls=int(k), cls_name=model.names.get(int(k), str(k)),
                         conf=float(cf)))
    return dets


def draw_detections(img, detections, crop_origin=(0, 0)):
    """Draw detection boxes + labels on img. Detections' bboxes are in full-frame
    coords; crop_origin subtracts to map them into a crop's local coords."""
    ox, oy = crop_origin
    out = img.copy()
    for d in detections:
        x1, y1, x2, y2 = d['bbox']
        x1, y1, x2, y2 = x1 - ox, y1 - oy, x2 - ox, y2 - oy
        color = PALETTE[d['cls'] % len(PALETTE)]
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        tag = f"{d['cls_name']} {d['conf']:.2f}"
        if 'cluster_depth' in d:
            tag += f" | {d['cluster_depth']:.2f}m"
        (tw, th), _ = cv2.getTextSize(tag, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
        cv2.rectangle(out, (x1, y1 - th - 8), (x1 + tw + 6, y1), color, -1)
        cv2.putText(out, tag, (x1 + 3, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 2, cv2.LINE_AA)
    return out


def draw_cluster_bboxes(img, kept, dropped):
    out = img.copy()
    for c in dropped:
        x1, y1, x2, y2 = c['bbox']
        cv2.rectangle(out, (x1, y1), (x2, y2), (80, 80, 180), 1)
    for i, c in enumerate(kept):
        x1, y1, x2, y2 = c['bbox']
        color = PALETTE[i % len(PALETTE)]
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        cv2.putText(out, f"z={c['median_depth']:.1f}m  n={c['n_points']}",
                    (x1, max(y1 - 6, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                    color, 2, cv2.LINE_AA)
    return out


def save_in_capture_dir(img, name):
    """Write img to STATE['capture_dir'] as <name>.png."""
    d = STATE.get('capture_dir')
    if d is None:
        raise RuntimeError('no capture_dir yet — run Step 1 first.')
    path = d / f'{name}.png'
    cv2.imwrite(str(path), img)
    print(f'  saved {path.relative_to(ANALYSIS_DIR.parent)}')
    return path


## Step 1 — Capture raw

Opens the ZED, grabs one (or `BURST_COUNT`) frames, **closes the ZED** immediately,
creates `analysis/depth_view/<timestamp>/`, saves `raw.png`. The timestamp here
is the *only* timestamp used by every later step.


In [ ]:
zed = open_zed()
runtime   = sl.RuntimeParameters()
image_mat = sl.Mat()
depth_mat = sl.Mat()

# Lock the timestamp + folder up front.
STATE['timestamp']   = datetime.now().strftime('%Y%m%d_%H%M%S')
STATE['capture_dir'] = ANALYSIS_DIR / STATE['timestamp']
STATE['capture_dir'].mkdir(parents=True, exist_ok=True)
print(f'capture dir: {STATE["capture_dir"]}')

try:
    if SHUTTER_MODE == 'single':
        rgb, depth = _grab(zed, runtime, image_mat, depth_mat)
        STATE['rgb'], STATE['depth'] = rgb, depth
        print(f'  single: {rgb.shape} RGB + {depth.shape} depth')
    elif SHUTTER_MODE == 'burst':
        STATE['rgbs'], STATE['depths'] = [], []
        for _ in range(BURST_COUNT):
            r, d = _grab(zed, runtime, image_mat, depth_mat)
            STATE['rgbs'].append(r); STATE['depths'].append(d)
        idx = max(0, min(FRAME_INDEX, BURST_COUNT - 1))
        STATE['rgb'], STATE['depth'] = STATE['rgbs'][idx], STATE['depths'][idx]
        print(f'  burst: {BURST_COUNT} frames captured; using FRAME_INDEX={idx}')
    else:
        raise ValueError(f'SHUTTER_MODE must be "single" or "burst", got {SHUTTER_MODE!r}')
finally:
    zed.close()
    print('  ZED closed')

save_in_capture_dir(STATE['rgb'], 'raw')


## Step 2 — Full-frame YOLO

YOLO on the raw RGB, no masking. Saves `yolo.png`. Loads the model into
`STATE['model']` so Step 4 reuses it instead of reloading.


In [ ]:
if 'rgb' not in STATE:
    print('run Step 1 first.')
else:
    if 'model' not in STATE:
        STATE['model'] = load_model()
    dets = predict_on_image(STATE['model'], STATE['rgb'])
    STATE['full_detections'] = dets
    annotated = draw_detections(STATE['rgb'], dets)
    save_in_capture_dir(annotated, 'yolo')
    print(f'  full-frame detections: {len(dets)}')
    for d in dets:
        print(f"    {d['cls_name']} conf={d['conf']:.2f}")


## Step 3 — DBSCAN overlay

Runs DBSCAN, draws kept clusters (coloured) and dropped-as-wall clusters
(faded red) on the RGB. Saves `dbscan.png`. Re-runnable — tune `DBSCAN_EPS`,
`WALL_DEPTH_M`, or `DEPTH_DOWNSAMPLE_STEP` in Cell 2 and re-run this cell alone.


In [ ]:
if 'rgb' not in STATE:
    print('run Step 1 first.')
else:
    points, labels = cluster_depth(STATE['depth'])
    kept, dropped  = filter_clusters(points, labels)
    STATE.update(points=points, labels=labels,
                 kept_clusters=kept, dropped_clusters=dropped)
    annotated = draw_cluster_bboxes(STATE['rgb'], kept, dropped)
    save_in_capture_dir(annotated, 'dbscan')
    print(f'  kept={len(kept)}  dropped={len(dropped)}')
    for c in kept:
        print(f"    cluster {c['label']}  n={c['n_points']}  median_depth={c['median_depth']:.2f}m")


## Step 4 — Per-cluster masked inference

For every kept cluster:

1. Rasterise the cluster's sparse points into a full-resolution pixel mask
   (dilated by `DEPTH_DOWNSAMPLE_STEP` so the mask fills the gaps between samples).
2. Apply that mask to the RGB — non-cluster pixels are set to pitch black.
3. Crop to the cluster's bbox.
4. Run YOLO on the masked crop.
5. Save `cluster_inference_{i}.png` (one file per cluster).

Stashes all per-cluster results + masks in `STATE['per_cluster']` so Step 5 can
stitch them without re-running inference.


In [ ]:
if 'rgb' not in STATE or 'kept_clusters' not in STATE:
    print('run Steps 1 and 3 first.')
else:
    if 'model' not in STATE:
        STATE['model'] = load_model()
    H, W = STATE['rgb'].shape[:2]
    per_cluster = []
    for i, cl in enumerate(STATE['kept_clusters']):
        x1, y1, x2, y2 = cl['bbox']
        # 1-2: full-frame mask + blackout
        mask = cluster_pixel_mask(STATE['points'], STATE['labels'],
                                   cl['label'], (H, W), DEPTH_DOWNSAMPLE_STEP)
        masked_full = STATE['rgb'].copy()
        masked_full[mask == 0] = 0
        # 3: crop
        crop = masked_full[y1:y2, x1:x2]
        # 4: YOLO
        dets = predict_on_image(STATE['model'], crop, crop_origin=(x1, y1))
        for d in dets:
            d['cluster_depth'] = cl['median_depth']
            d['cluster_label'] = cl['label']
        # draw on the crop (remapped)
        annotated_crop = draw_detections(crop, dets, crop_origin=(x1, y1))
        save_in_capture_dir(annotated_crop, f'cluster_inference_{i}')
        per_cluster.append(dict(cluster=cl, mask=mask, crop=crop,
                                annotated_crop=annotated_crop, detections=dets))
        print(f'  cluster {cl["label"]} (n={cl["n_points"]}, '
              f'z={cl["median_depth"]:.2f}m): {len(dets)} detection(s)')
        for d in dets:
            print(f"      {d['cls_name']} conf={d['conf']:.2f}")
    STATE['per_cluster'] = per_cluster
    print(f'  total clusters processed: {len(per_cluster)}')


## Step 5 — Depth-aware stitched inference

Reconstructs the full-resolution frame from the per-cluster masked results.
Pixels inside any kept cluster's mask retain their RGB values; everything else
is pitch black. All per-cluster YOLO detections are drawn on top at their
original full-frame coordinates. Saves `depth_aware_inference.png` — this is
the A/B counterpart to `yolo.png`.


In [ ]:
if 'rgb' not in STATE or 'per_cluster' not in STATE:
    print('run Steps 1-4 first.')
else:
    H, W = STATE['rgb'].shape[:2]
    canvas = np.zeros_like(STATE['rgb'])
    for res in STATE['per_cluster']:
        m = res['mask'] > 0
        canvas[m] = STATE['rgb'][m]
    all_dets = [d for res in STATE['per_cluster'] for d in res['detections']]
    canvas = draw_detections(canvas, all_dets)
    STATE['depth_aware_canvas'] = canvas
    save_in_capture_dir(canvas, 'depth_aware_inference')
    print(f'  stitched canvas: {len(all_dets)} detection(s) across '
          f'{len(STATE["per_cluster"])} cluster(s)')


## Step 6 — 3D DBSCAN scatter *(optional)*

Plotly 3D scatter of the point cloud — kept clusters coloured, dropped
(wall / noise) clusters faded. Gated by `VISUALIZE_CLUSTERS_3D` in Cell 2.
Saves `3d.html` in the same capture folder as the PNGs.


In [ ]:
if not VISUALIZE_CLUSTERS_3D:
    print('VISUALIZE_CLUSTERS_3D = False in Cell 2. Toggle True to render the 3D scatter.')
elif 'points' not in STATE:
    print('run Step 3 first.')
else:
    import plotly.graph_objects as go
    points, labels = STATE['points'], STATE['labels']
    kept_ids = {c['label'] for c in STATE.get('kept_clusters', [])}
    if len(points) == 0:
        print('  point cloud is empty.')
    else:
        pts = MinMaxScaler().fit_transform(points)
        PALETTE_3D = ['#4C6EF5', '#F03E3E', '#37B24D', '#F59F00',
                      '#AE3EC9', '#1098AD', '#D6336C', '#E8590C']
        traces = []
        for lbl in sorted(set(labels.tolist())):
            m = labels == lbl
            p = pts[m]
            if lbl == -1:
                name, color, size, opacity = 'Noise', 'rgba(180,180,180,0.25)', 1.5, 0.25
            elif lbl in kept_ids:
                name, color, size, opacity = f'Cluster {lbl} (kept)', PALETTE_3D[lbl % len(PALETTE_3D)], 3, 0.9
            else:
                name, color, size, opacity = f'Cluster {lbl} (wall)', 'rgba(90,90,90,0.45)', 2, 0.45
            traces.append(go.Scatter3d(
                x=p[:, 0], y=p[:, 1], z=p[:, 2], mode='markers',
                marker=dict(size=size, color=color, opacity=opacity), name=name))
        fig = go.Figure(traces)
        fig.update_layout(
            title=f'DBSCAN (eps={DBSCAN_EPS}, minPts={DBSCAN_MIN_SAMPLES}, '
                  f'wall>{WALL_DEPTH_M}m, step={DEPTH_DOWNSAMPLE_STEP})',
            scene=dict(xaxis_title='X (pixel)', yaxis_title='Y (pixel)',
                       zaxis_title='depth (m)',
                       aspectmode='manual', aspectratio=dict(x=1.8, y=1.0, z=0.6)),
            legend=dict(itemsizing='constant'),
            margin=dict(l=0, r=0, b=0, t=40))
        out = STATE['capture_dir'] / '3d.html'
        fig.write_html(str(out))
        print(f'  saved {out.name}')
